# Analiza strategije guma u Formuli 1

Ovaj notebook implementira kompletan tok rada: prikupljanje podataka iz FastF1 biblioteke, transformaciju podataka, primenu algoritama klasifikacije i klasterovanja, evaluaciju modela i čuvanje rezultata svakog run-a (sa timestamp-om i parametrima modela).

## 1. Učitavanje biblioteka i putanja

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_collection import CollectionConfig, collect_race_laps
from src.preprocessing import preprocess_laps
from src.modeling import evaluate_classification_models, evaluate_clustering_models
from src.run_pipeline import run_pipeline

raw_path = PROJECT_ROOT / "data/raw/f1_laps_raw.csv"
processed_path = PROJECT_ROOT / "data/processed/f1_laps_processed.csv"
results_path = PROJECT_ROOT / "results"

raw_path.parent.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)
results_path.mkdir(parents=True, exist_ok=True)

run_history_path = results_path / "run_history.csv"
latest_pipeline_run_id = None
if run_history_path.exists():
    run_history_df = pd.read_csv(run_history_path)
    if not run_history_df.empty and "run_timestamp" in run_history_df.columns:
        run_history_df["run_timestamp"] = pd.to_datetime(run_history_df["run_timestamp"], errors="coerce")
        latest_pipeline_run_id = run_history_df.sort_values("run_timestamp").iloc[-1]["run_id"]
print(f"Poslednji pipeline run_id: {latest_pipeline_run_id}")

## 2. Prikupljanje podataka (2019-2025)

In [ ]:
config = CollectionConfig(start_year=2019, end_year=2025, cache_dir=PROJECT_ROOT / "data/cache")
raw_laps = collect_race_laps(config)
raw_laps.to_csv(raw_path, index=False)

print(f"Učitano krugova (sirovi podaci): {len(raw_laps)}")
raw_laps.head()

## 3. Transformacija i filtriranje podataka

In [ ]:
processed_laps = preprocess_laps(raw_laps)
processed_laps.to_csv(processed_path, index=False)

print(f"Broj krugova nakon obrade: {len(processed_laps)}")
processed_laps.head(10)

## 4. Vizualizacija podataka

In [ ]:
compound_counts = processed_laps["Compound"].astype(str).value_counts()
compound_counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
compound_counts.sort_index().plot(kind="bar", ax=ax, color=["#e74c3c", "#f1c40f", "#2ecc71"])
ax.set_title("Raspodela tipova guma")
ax.set_xlabel("Tip gume")
ax.set_ylabel("Broj krugova")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for compound in ["SOFT", "MEDIUM", "HARD"]:
    subset = processed_laps.loc[processed_laps["Compound"].astype(str) == compound, "LapTime"]
    if not subset.empty:
        ax.hist(subset, bins=50, alpha=0.45, label=compound)

ax.set_title("Distribucija vremena kruga po tipu gume")
ax.set_xlabel("Vreme kruga (s)")
ax.set_ylabel("Broj krugova")
ax.legend()
plt.tight_layout()

In [ ]:
top_teams = processed_laps["CanonicalTeam"].astype(str).value_counts().head(10).index
box_data = [processed_laps.loc[processed_laps["CanonicalTeam"].astype(str) == team, "LapTime"].values for team in top_teams]

fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(box_data, tick_labels=top_teams, showfliers=False)
ax.set_title("Vreme kruga po kanonskom timu (bez outlier-a)")
ax.set_xlabel("Tim")
ax.set_ylabel("Vreme kruga (s)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
corr_cols = ["LapNumber", "TyreLife", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time", "MaxSpeed", "AvgSpeed"]
corr = processed_laps[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(corr_cols)))
ax.set_yticks(np.arange(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticklabels(corr_cols)
ax.set_title("Korelaciona matrica numeričkih obeležja")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

## 5. Klasifikacija tipa gume

In [ ]:
classification_metrics = evaluate_classification_models(
    processed_laps,
    output_dir=results_path / "classification",
    random_state=42,
)
classification_metrics

In [ ]:
classification_history_path = results_path / "classification/classification_metrics_history.csv"
if classification_history_path.exists():
    classification_history = pd.read_csv(classification_history_path)
    classification_history["run_timestamp"] = pd.to_datetime(classification_history["run_timestamp"], errors="coerce")
    if latest_pipeline_run_id is not None and latest_pipeline_run_id in set(classification_history["run_id"]):
        latest_classification_run_id = latest_pipeline_run_id
    else:
        latest_classification_run_id = classification_history.sort_values("run_timestamp").iloc[-1]["run_id"]
    classification_metrics_latest = classification_history[classification_history["run_id"] == latest_classification_run_id].copy()
else:
    latest_classification_run_id = classification_metrics["run_id"].iloc[-1] if "run_id" in classification_metrics.columns else None
    classification_metrics_latest = classification_metrics.copy()

print(f"Poslednji run klasifikacije: {latest_classification_run_id}")
classification_metrics_latest

### 5.1 Vizualizacija evaluacije klasifikacije (poslednji run)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for idx, metric_col in enumerate(["macro_f1", "weighted_f1"]):
    pivot = classification_metrics_latest.pivot(index="model", columns="split", values=metric_col)
    pivot.plot(kind="bar", ax=axes[idx])
    axes[idx].set_title(f"{metric_col} po modelu")
    axes[idx].set_xlabel("Model")
    axes[idx].set_ylabel(metric_col)
    axes[idx].set_ylim(0, 1)
    axes[idx].legend(title="Skup")
plt.tight_layout()

In [ ]:
if latest_classification_run_id is not None:
    cm_files = sorted((results_path / "classification/runs").glob(f"confusion_matrix_test_*__{latest_classification_run_id}.csv"))
else:
    cm_files = sorted((results_path / "classification").glob("confusion_matrix_test_*.csv"))

if not cm_files:
    print("Nema confusion matrix fajlova za poslednji run.")

for cm_file in cm_files:
    cm_df = pd.read_csv(cm_file, index_col=0)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_df.values, cmap="Blues")
    ax.set_xticks(np.arange(cm_df.shape[1]))
    ax.set_yticks(np.arange(cm_df.shape[0]))
    ax.set_xticklabels(cm_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(cm_df.index)

    threshold = cm_df.values.max() * 0.5 if cm_df.values.size else 0
    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            value = int(cm_df.iat[i, j])
            color = "white" if value > threshold else "black"
            ax.text(j, i, str(value), ha="center", va="center", color=color, fontsize=10)

    model_label = cm_file.stem.replace("confusion_matrix_test_", "")
    model_label = model_label.split("__")[0]
    ax.set_title(f"Confusion Matrix ({model_label})")
    ax.set_xlabel("Predviđena klasa")
    ax.set_ylabel("Stvarna klasa")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## 6. Klasterovanje krugova

In [ ]:
clustering_metrics = evaluate_clustering_models(
    processed_laps,
    output_dir=results_path / "clustering",
    random_state=42,
)
clustering_metrics

In [ ]:
clustering_history_path = results_path / "clustering/clustering_metrics_history.csv"
if clustering_history_path.exists():
    clustering_history = pd.read_csv(clustering_history_path)
    clustering_history["run_timestamp"] = pd.to_datetime(clustering_history["run_timestamp"], errors="coerce")
    if latest_pipeline_run_id is not None and latest_pipeline_run_id in set(clustering_history["run_id"]):
        latest_clustering_run_id = latest_pipeline_run_id
    else:
        latest_clustering_run_id = clustering_history.sort_values("run_timestamp").iloc[-1]["run_id"]
    clustering_metrics_latest = clustering_history[clustering_history["run_id"] == latest_clustering_run_id].copy()
else:
    latest_clustering_run_id = clustering_metrics["run_id"].iloc[-1] if "run_id" in clustering_metrics.columns else None
    clustering_metrics_latest = clustering_metrics.copy()

print(f"Poslednji run klasterovanja: {latest_clustering_run_id}")
clustering_metrics_latest

### 6.1 Vizualizacija evaluacije klasterovanja (poslednji run)

In [ ]:
plot_cols = ["adjusted_rand_index", "cluster_purity"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for idx, metric_col in enumerate(plot_cols):
    axes[idx].bar(clustering_metrics_latest["model"], clustering_metrics_latest[metric_col], color=["#3498db", "#16a085"])
    axes[idx].set_title(f"{metric_col} po modelu")
    axes[idx].set_xlabel("Model")
    axes[idx].set_ylabel(metric_col)
    axes[idx].set_ylim(0, 1)

plt.tight_layout()

In [ ]:
if latest_clustering_run_id is not None:
    assignment_files = sorted((results_path / "clustering/runs").glob(f"cluster_assignments_*__{latest_clustering_run_id}.csv"))
else:
    assignment_files = sorted((results_path / "clustering").glob("cluster_assignments_*.csv"))

for assignment_file in assignment_files:
    assign_df = pd.read_csv(assignment_file)
    if "CanonicalTeam" not in assign_df.columns:
        continue

    contingency = pd.crosstab(assign_df["CanonicalTeam"], assign_df["cluster"], normalize="index")

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(contingency.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(np.arange(contingency.shape[1]))
    ax.set_yticks(np.arange(contingency.shape[0]))
    ax.set_xticklabels(contingency.columns)
    ax.set_yticklabels(contingency.index)
    ax.set_xlabel("Klaster")
    ax.set_ylabel("Kanonski tim")
    ax.set_title(f"Učešće timova po klasterima ({assignment_file.stem})")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Udeo")
    plt.tight_layout()
    plt.show()

## 7. Istorija svih run-ova i parametri modela

In [ ]:
saved_classification_history = pd.read_csv(results_path / "classification/classification_metrics_history.csv")
saved_clustering_history = pd.read_csv(results_path / "clustering/clustering_metrics_history.csv")

saved_classification_history = saved_classification_history.sort_values("run_timestamp")
saved_clustering_history = saved_clustering_history.sort_values("run_timestamp")

print("Klasifikacija - poslednjih 10 zapisa:")
display(saved_classification_history.tail(10))

print("Klasterovanje - poslednjih 10 zapisa:")
display(saved_clustering_history.tail(10))

## 8. Pokretanje celog pipeline-a jednom komandom

Napomena: svaki run sada dobija `run_id` i `run_timestamp`, i automatski se dodaje u history fajlove.

In [ ]:
summary = run_pipeline(
    start_year=2019,
    end_year=2025,
    raw_output_path=raw_path,
    processed_output_path=processed_path,
    results_dir=results_path,
    cache_dir=PROJECT_ROOT / "data/cache",
    random_state=42,
)
summary